In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from grid_cells.random_walk import generate_bat_flight
from grid_cells.bat_hd_system_alt import BatHeadDirectionSystem, sphere_to_toroid
from grid_cells.plot_tools import angular_error, smooth_ts
import os
from concurrent.futures import ThreadPoolExecutor

DATA_DIR = "../simulation_data/hebb_alt"
PLOTS_DIR = "../plots/hebb_alt"
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

In [ ]:
T = 400
dt = 0.5e-3
n = 256
intrinsic_noise = 0.01
input_noise = 1
n_conj = 20
interval = 100
bat_flight = generate_bat_flight(T=T, dt=dt)


net_no_conj_connection = BatHeadDirectionSystem(
    input_noise=input_noise,
    intrinsic_noise=intrinsic_noise,
    n_conjunctive=n_conj,
    forward_strength=0,
    anchor_strength=0,
)

net = BatHeadDirectionSystem(
    input_noise=input_noise,
    intrinsic_noise=intrinsic_noise,
    n_conjunctive=n_conj,
)
net_anchor = BatHeadDirectionSystem(
    input_noise=input_noise,
    intrinsic_noise=intrinsic_noise,
    n_conjunctive=n_conj,
)
net_anchor_no_fwd = BatHeadDirectionSystem(
    input_noise=input_noise,
    intrinsic_noise=intrinsic_noise,
    n_conjunctive=n_conj,
    connect_vc_directly=True,
)


net_no_conj_connection.warm_up(initial_dir=bat_flight["dir_torus"][0])
net.warm_up(initial_dir=bat_flight["dir_torus"][0])
net_anchor.warm_up(initial_dir=bat_flight["dir_torus"][0])
net_anchor_no_fwd.warm_up(initial_dir=bat_flight["dir_torus"][0])

n_steps = bat_flight["pos"].shape[0]

with ThreadPoolExecutor(max_workers=4) as executor:
    no_anchor_future = executor.submit(
        net.run_simulation, bat_flight["dir_vel"], interval=interval, verbose=False
    )
    no_anchor_no_conj_future = executor.submit(
        net_no_conj_connection.run_simulation,
        bat_flight["dir_vel"],
        interval=interval,
        verbose=False,
    )

    anchor_future = executor.submit(
        net_anchor.run_simulation,
        bat_flight["dir_vel"],
        dir=bat_flight["dir_sphere"],
        save_weights=True,
        interval=interval,
        verbose=False,
    )

    anchor_no_fwd_future = executor.submit(
        net_anchor_no_fwd.run_simulation,
        bat_flight["dir_vel"],
        dir=bat_flight["dir_sphere"],
        save_weights=True,
        interval=interval,
        verbose=True,
    )

    recording_no_conj = no_anchor_no_conj_future.result()
    recording = no_anchor_future.result()
    recording_anchor = anchor_future.result()
    recording_anchor_no_fwd = anchor_no_fwd_future.result()

In [ ]:
np.savez(os.path.join(DATA_DIR, "recording"), **recording)
np.savez(os.path.join(DATA_DIR, "recording_anchor"), **recording_anchor)
np.savez(os.path.join(DATA_DIR, "recording_anchor_no_fwd"), **recording_anchor_no_fwd)

In [ ]:
np.mean(
    np.abs(
        angular_error(
            recording_anchor["decoded_angle"], bat_flight["dir_torus"][::interval]
        )
    ),
    axis=0,
), np.mean(
    np.abs(
        angular_error(
            recording_anchor_no_fwd["decoded_angle"],
            bat_flight["dir_torus"][::interval],
        )
    ),
    axis=0,
), np.mean(
    np.abs(
        angular_error(recording["decoded_angle"], bat_flight["dir_torus"][::interval])
    ),
    axis=0,
), np.mean(
    np.abs(
        angular_error(
            recording_no_conj["decoded_angle"],
            bat_flight["dir_torus"][::interval],
        )
    ),
    axis=0,
)

In [ ]:
smooth_interval = 10
fig, ax = plt.subplots(
    nrows=2,
    figsize=(10, 8),
    sharex=True,
    gridspec_kw={"hspace": (0.05)},
)
for angle_iter, angle in enumerate(["Azimuth", "Pitch"]):
    simple_anchor_error = angular_error(
        recording_anchor["decoded_angle"][:, angle_iter],
        bat_flight["dir_torus"][::interval, angle_iter],
    )
    no_fwd_anchor_error = angular_error(
        recording_anchor_no_fwd["decoded_angle"][:, angle_iter],
        bat_flight["dir_torus"][::interval, angle_iter],
    )
    error = angular_error(
        recording["decoded_angle"][:, angle_iter],
        bat_flight["dir_torus"][::interval, angle_iter],
    )
    error_no_conj = angular_error(
        recording_no_conj["decoded_angle"][:, angle_iter],
        bat_flight["dir_torus"][::interval, angle_iter],
    )
    ax[1 * angle_iter].plot(
        smooth_ts(bat_flight["time"][::interval], smooth_interval),
        np.abs(smooth_ts(simple_anchor_error, smooth_interval)),
        label="Anchoring",
    )
    ax[1 * angle_iter].plot(
        smooth_ts(bat_flight["time"][::interval], smooth_interval),
        np.abs(smooth_ts(no_fwd_anchor_error, smooth_interval)),
        label="Anchoring (No FWD)",
    )
    ax[1 * angle_iter].plot(
        smooth_ts(bat_flight["time"][::interval], smooth_interval),
        np.abs(smooth_ts(error, smooth_interval)),
        label="No Anchoring",
        ls="--",
    )
    ax[1 * angle_iter].plot(
        smooth_ts(bat_flight["time"][::interval], smooth_interval),
        np.abs(smooth_ts(error_no_conj, smooth_interval)),
        label="No Anchoring, No Conj. Cell",
        ls="--",
    )

    ax[1 * angle_iter].set_ylabel(angle + " Error")

for axis_iter in range(len(ax) - 1):
    ax[axis_iter].get_xaxis().set_visible(False)


ax[0].legend()
ax[-1].set_xlabel("Time [s]")
fig.savefig(os.path.join(PLOTS_DIR, "bat_flight_anchoring_effect.svg"))

In [ ]:
np.max(recording_no_conj["conj_cells"]), np.max(recording["conj_cells"]), np.max(
    recording_anchor["conj_cells"]
)

In [ ]:
np.max(recording_anchor_no_fwd["conj_cells"]), np.max(
    recording_anchor_no_fwd["visual_trace"]
)

In [ ]:
from grid_cells.plot_tools import activity_map

cell_index = 1
fig, ax = plt.subplots(ncols=3, sharey=True, figsize=(9, 4))
activity, counts, direction_edges = activity_map(
    bat_flight["dir_torus"][::interval],
    recording_anchor["yaw_cells"][:, (cell_index + n // 2) % n],
    nbins=15,
    sigma=1,
)
ax[0].imshow(
    activity.T,
    extent=(
        direction_edges[0][0] / np.pi,
        direction_edges[0][-1] / np.pi,
        direction_edges[1][0] / np.pi,
        direction_edges[1][-1] / np.pi,
    ),
    origin="lower",
    interpolation="bilinear",
)
ax[0].set_title("Yaw")
activity, counts, direction_edges = activity_map(
    bat_flight["dir_torus"][::interval],
    recording_anchor["pitch_cells"][:, cell_index],
    nbins=15,
    sigma=1,
)

ax[1].imshow(
    activity.T,
    extent=(
        direction_edges[0][0] / np.pi,
        direction_edges[0][-1] / np.pi,
        direction_edges[1][0] / np.pi,
        direction_edges[1][-1] / np.pi,
    ),
    origin="lower",
    interpolation="bilinear",
)
ax[1].set_title("Pitch")

activity, counts, direction_edges = activity_map(
    bat_flight["dir_torus"][::interval],
    recording_anchor["conj_cells"][:, cell_index],
    nbins=15,
    sigma=1,
)
ax[2].imshow(
    activity.T,
    extent=(
        direction_edges[0][0] / np.pi,
        direction_edges[0][-1] / np.pi,
        direction_edges[1][0] / np.pi,
        direction_edges[1][-1] / np.pi,
    ),
    origin="lower",
    interpolation="bilinear",
)
ax[2].set_title("Conjunctive")
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "bat_flight_cell_types_anchor.svg"))

In [ ]:
from grid_cells.plot_tools import activity_map

cell_index = 1
fig, ax = plt.subplots(ncols=2, nrows=2, sharey=True)
ax = ax.flatten()
activity, counts, direction_edges = activity_map(
    bat_flight["dir_torus"][::interval],
    recording_anchor_no_fwd["yaw_cells"][:, (cell_index + n // 2) % n],
    nbins=15,
    sigma=1,
)
ax[0].imshow(
    activity.T,
    extent=(
        direction_edges[0][0] / np.pi,
        direction_edges[0][-1] / np.pi,
        direction_edges[1][0] / np.pi,
        direction_edges[1][-1] / np.pi,
    ),
    origin="lower",
    interpolation="bilinear",
)
ax[0].set_title("Yaw")
activity, counts, direction_edges = activity_map(
    bat_flight["dir_torus"][::interval],
    recording_anchor_no_fwd["pitch_cells"][:, cell_index],
    nbins=15,
    sigma=1,
)

ax[1].imshow(
    activity.T,
    extent=(
        direction_edges[0][0] / np.pi,
        direction_edges[0][-1] / np.pi,
        direction_edges[1][0] / np.pi,
        direction_edges[1][-1] / np.pi,
    ),
    origin="lower",
    interpolation="bilinear",
)
ax[1].set_title("Pitch")

activity, counts, direction_edges = activity_map(
    bat_flight["dir_torus"][::interval],
    recording_anchor_no_fwd["conj_cells"][:, cell_index],
    nbins=15,
    sigma=1,
)
ax[2].imshow(
    activity.T,
    extent=(
        direction_edges[0][0] / np.pi,
        direction_edges[0][-1] / np.pi,
        direction_edges[1][0] / np.pi,
        direction_edges[1][-1] / np.pi,
    ),
    origin="lower",
    interpolation="bilinear",
)
ax[2].set_title("Conjunctive")


activity, counts, direction_edges = activity_map(
    bat_flight["dir_torus"][::interval],
    recording_anchor_no_fwd["visual_trace"][:, cell_index],
    nbins=15,
    sigma=1,
)
ax[3].imshow(
    activity.T,
    extent=(
        direction_edges[0][0] / np.pi,
        direction_edges[0][-1] / np.pi,
        direction_edges[1][0] / np.pi,
        direction_edges[1][-1] / np.pi,
    ),
    origin="lower",
    interpolation="bilinear",
)
ax[3].set_title("Visual Ancor")
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "bat_flight_cell_types_anchor_no_fwd.svg"))

In [ ]:
from grid_cells.plot_tools import activity_map

cell_index = 1
fig, ax = plt.subplots(ncols=3, sharey=True, figsize=(9, 4))
activity, counts, direction_edges = activity_map(
    bat_flight["dir_torus"][::interval],
    recording["yaw_cells"][:, 100],
    nbins=15,
    sigma=1,
)
ax[0].imshow(
    activity.T,
    extent=(
        direction_edges[0][0] / np.pi,
        direction_edges[0][-1] / np.pi,
        direction_edges[1][0] / np.pi,
        direction_edges[1][-1] / np.pi,
    ),
    origin="lower",
    interpolation="bilinear",
)
ax[0].set_title("Yaw")
activity, counts, direction_edges = activity_map(
    bat_flight["dir_torus"][::interval],
    recording["pitch_cells"][:, cell_index],
    nbins=15,
    sigma=1,
)

ax[1].imshow(
    activity.T,
    extent=(
        direction_edges[0][0] / np.pi,
        direction_edges[0][-1] / np.pi,
        direction_edges[1][0] / np.pi,
        direction_edges[1][-1] / np.pi,
    ),
    origin="lower",
    interpolation="bilinear",
)
ax[1].set_title("Pitch")

activity, counts, direction_edges = activity_map(
    bat_flight["dir_torus"][::interval],
    recording["conj_cells"][:, cell_index],
    nbins=15,
    sigma=1,
)
ax[2].imshow(
    activity.T,
    extent=(
        direction_edges[0][0] / np.pi,
        direction_edges[0][-1] / np.pi,
        direction_edges[1][0] / np.pi,
        direction_edges[1][-1] / np.pi,
    ),
    origin="lower",
    interpolation="bilinear",
)
ax[2].set_title("Conjunctive")
fig.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, "bat_flight_cell_types.svg"))